In [1]:
REPO_URL = "https://github.com/MberkKeskin/turkish_legal_rag_assistant.git"
PROJECT_DIR = "/content/turkish_legal_rag_assistant/FINAL_SUBMISSION"

HF_MODELS = {
    "bge-m3-legal-ft-system2": "Berk2003/bge-m3-legal-ft-system2",
    "bge_reranker_legal_ft_v4_error_mined_hf": "Berk2003/bge-reranker-legal-ft-v4-error-mined-hf",
    "qwen2_5_3b_legal_lora_sft_faithful_v2_final": "Berk2003/qwen2-5-3b-legal-lora-sft-faithful-v2-final",
}

print("Repository:", REPO_URL)
print("Project dir:", PROJECT_DIR)
print("Hugging Face models:")
for local_name, repo_id in HF_MODELS.items():
    print(f"  {local_name} <- {repo_id}")

!nvidia-smi || true

Repository: https://github.com/MberkKeskin/turkish_legal_rag_assistant.git
Project dir: /content/turkish_legal_rag_assistant/FINAL_SUBMISSION
Hugging Face models:
  bge-m3-legal-ft-system2 <- Berk2003/bge-m3-legal-ft-system2
  bge_reranker_legal_ft_v4_error_mined_hf <- Berk2003/bge-reranker-legal-ft-v4-error-mined-hf
  qwen2_5_3b_legal_lora_sft_faithful_v2_final <- Berk2003/qwen2-5-3b-legal-lora-sft-faithful-v2-final
Sat Jun 13 11:23:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG 

In [2]:
%cd /content
!rm -rf turkish_legal_rag_assistant
!git clone "$REPO_URL"
%cd "$PROJECT_DIR"

!pwd
!ls

/content
Cloning into 'turkish_legal_rag_assistant'...
remote: Enumerating objects: 439, done.
remote: Counting objects: 100% (439/439), done.
remote: Compressing objects: 100% (274/274), done.
remote: Total 439 (delta 182), reused 404 (delta 161), pack-reused 0 (from 0)
Receiving objects: 100% (439/439), 27.28 MiB | 12.70 MiB/s, done.
Resolving deltas: 100% (182/182), done.
Updating files: 100% (398/398), done.
/content/turkish_legal_rag_assistant/FINAL_SUBMISSION
/content/turkish_legal_rag_assistant/FINAL_SUBMISSION
 allTestResults				 FINAL_LEGAL_RAG_SYSTEM
 app					 final_ui.py
 data					 models
 evaluate_base_vs_final_retrieval.py	'README (1).md'
 evaluate_custom_benchmark.py		 README.md
 evaluate_one_physical_stage_worker.py	 requirements.txt
 evaluate_physical_stagewise_20.py	 smoke_test_light.py
 evaluate_real_stagewise_50.py		 smoke_test.py
 evaluate_stagewise_full_20.py		 testResultsFinal
 evaluation_results


In [3]:
!pip install -q -r requirements.txt
!pip install -q -U huggingface_hub

In [4]:
from pathlib import Path
from huggingface_hub import snapshot_download
import shutil
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

models_dir = Path(PROJECT_DIR) / "models"

# GitHub'dan eksik/boş models klasörü geldiyse temizle
if models_dir.exists():
    shutil.rmtree(models_dir)

models_dir.mkdir(parents=True, exist_ok=True)

def count_files(path):
    path = Path(path)
    return len([x for x in path.rglob("*") if x.is_file()]) if path.exists() else 0

def folder_size_mb(path):
    path = Path(path)
    return sum(x.stat().st_size for x in path.rglob("*") if x.is_file()) / (1024 * 1024) if path.exists() else 0

for local_name, repo_id in HF_MODELS.items():
    target_dir = models_dir / local_name

    print("\n" + "=" * 90)
    print("Downloading model")
    print("HF repo   :", repo_id)
    print("Local path:", target_dir)
    print("=" * 90)

    snapshot_download(
        repo_id=repo_id,
        repo_type="model",
        local_dir=str(target_dir),
        local_dir_use_symlinks=False,
    )

    print("Downloaded:", local_name)
    print("Files:", count_files(target_dir))
    print("Size MB:", round(folder_size_mb(target_dir), 2))

print("\nFinal model check:")
for local_name in HF_MODELS:
    p = models_dir / local_name
    print(f"{local_name:60s} exists={p.exists()} files={count_files(p)} size_mb={round(folder_size_mb(p), 2)}")

missing = [
    name for name in HF_MODELS
    if not (models_dir / name).exists() or count_files(models_dir / name) == 0
]

if missing:
    raise RuntimeError(f"Missing model folders: {missing}")

print("\nAll Hugging Face models downloaded successfully.")


HF repo   : Berk2003/bge-m3-legal-ft-system2
Local path: /content/turkish_legal_rag_assistant/FINAL_SUBMISSION/models/bge-m3-legal-ft-system2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Downloaded: bge-m3-legal-ft-system2
Files: 22
Size MB: 2182.2

HF repo   : Berk2003/bge-reranker-legal-ft-v4-error-mined-hf
Local path: /content/turkish_legal_rag_assistant/FINAL_SUBMISSION/models/bge_reranker_legal_ft_v4_error_mined_hf


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Downloaded: bge_reranker_legal_ft_v4_error_mined_hf
Files: 20
Size MB: 2181.9

HF repo   : Berk2003/qwen2-5-3b-legal-lora-sft-faithful-v2-final
Local path: /content/turkish_legal_rag_assistant/FINAL_SUBMISSION/models/qwen2_5_3b_legal_lora_sft_faithful_v2_final


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Downloaded: qwen2_5_3b_legal_lora_sft_faithful_v2_final
Files: 22
Size MB: 68.08

Final model check:
bge-m3-legal-ft-system2                                      exists=True files=22 size_mb=2182.2
bge_reranker_legal_ft_v4_error_mined_hf                      exists=True files=20 size_mb=2181.9
qwen2_5_3b_legal_lora_sft_faithful_v2_final                  exists=True files=22 size_mb=68.08

All Hugging Face models downloaded successfully.


In [5]:
from pathlib import Path

checks = {
    "models/bge-m3-legal-ft-system2": ["model.safetensors"],
    "models/bge_reranker_legal_ft_v4_error_mined_hf": ["model.safetensors"],
    "models/qwen2_5_3b_legal_lora_sft_faithful_v2_final": ["adapter_model.safetensors"],
}

def size_mb(path):
    return path.stat().st_size / (1024 * 1024)

for folder, required_files in checks.items():
    folder_path = Path(folder)
    print("\n" + "=" * 80)
    print(folder)
    print("exists:", folder_path.exists())

    for req in required_files:
        matches = list(folder_path.rglob(req)) if folder_path.exists() else []
        if not matches:
            raise RuntimeError(f"Missing required file: {folder}/{req}")

        for m in matches:
            print(req, "->", m, "| size MB:", round(size_mb(m), 2))

print("\nRequired model files are valid.")


models/bge-m3-legal-ft-system2
exists: True
model.safetensors -> models/bge-m3-legal-ft-system2/model.safetensors | size MB: 2165.86

models/bge_reranker_legal_ft_v4_error_mined_hf
exists: True
model.safetensors -> models/bge_reranker_legal_ft_v4_error_mined_hf/model.safetensors | size MB: 2165.86

models/qwen2_5_3b_legal_lora_sft_faithful_v2_final
exists: True
adapter_model.safetensors -> models/qwen2_5_3b_legal_lora_sft_faithful_v2_final/adapter_model.safetensors | size MB: 57.16

Required model files are valid.


In [6]:
from pathlib import Path

required_items = [
    "README.md",
    "requirements.txt",
    "final_ui.py",
    "smoke_test.py",
    "smoke_test_light.py",
    "evaluate_custom_benchmark.py",
    "app",
    "data",
    "models",
]

print("Project file check:")
for item in required_items:
    p = Path(item)
    print(f"{item:40s}", "OK" if p.exists() else "MISSING")

Project file check:
README.md                                OK
requirements.txt                         OK
final_ui.py                              OK
smoke_test.py                            OK
smoke_test_light.py                      OK
evaluate_custom_benchmark.py             OK
app                                      OK
data                                     OK
models                                   OK


In [7]:
exec(open("final_ui.py", encoding="utf-8").read())

Output()

Dropdown(description='Örnek:', layout=Layout(width='100%'), options=('Kiracı kira bedelini her ay ne zaman öde…

Textarea(value='Kiracı kira bedelini her ay ne zaman ödemekle yükümlüdür?', description='Soru:', layout=Layout…

Button(button_style='primary', description='Cevap Üret', layout=Layout(height='42px', width='180px'), style=Bu…

Output()

FileUpload(value={}, accept='.txt,.pdf,.docx', description='Belge Yükle')

Textarea(value='Bu belgeye göre temel hüküm veya sonuç nedir?', description='Soru:', layout=Layout(height='95p…

Button(button_style='success', description='Belgeden Cevap Üret', layout=Layout(height='42px', width='230px'),…

Output()